# The KuaiRandDataset

Data analysis and observations

**Note:** Go to the bottom to see preprocessing steps

In [ ]:
# SPECIFY YOUR OWN DATA PATH
data_path = '../../../dataset/kuairand/KuaiRand-Pure/data/'
# data_path = '../../dataset/kuairand/KuaiRand-1K/data/'
# data_path = '~/public/kuairanddata/KuaiRand-27K/data/'

In [ ]:
import pandas as pd
# df_rand = pd.read_csv(data_path + 'log_standard_4_08_to_4_21_1k.csv')
# df_rand_2 = pd.read_csv(data_path + 'log_standard_4_22_to_5_08_1k.csv')
df_rand = pd.read_csv('dataset/kuairand/kuairand-Pure/data/log_standard_4_08_to_4_21_pure.csv')
df_rand_2 = pd.read_csv('dataset/kuairand/kuairand-Pure/data/log_standard_4_22_to_5_08_pure.csv')
df_rand = pd.concat([df_rand, df_rand_2], axis=0, ignore_index=True)
df_rand

In [ ]:
# example user
A = df_rand[(df_rand['user_id'] == 3)]
A[:20]

In [ ]:
print(df_rand.dtypes)

In [ ]:
df_vid = pd.read_csv('dataset/kuairand/kuairand-Pure/data/video_features_basic_pure.csv')
print(df_vid.dtypes)

In [ ]:
df_vid2 = pd.read_csv('dataset/kuairand/kuairand-Pure/data/video_features_statistic_pure.csv')
print(df_vid2.dtypes)

In [ ]:
df_vid[:2]

In [ ]:
df_vid2[:2]

In [ ]:
# n_user, n_item, n_record
print(len(df_rand['user_id'].unique()), len(df_rand['video_id'].unique()), len(df_rand))

In [ ]:
# user history length
counts = df_rand['user_id'].value_counts()
print(counts)

## Example User's Interaction Density

In [ ]:
users = df_rand['user_id'].unique()

In [ ]:
import matplotlib.pyplot as plt
from datetime import datetime as dt
import seaborn as sns
import numpy as np

min_hist_len = 200 # for kuairand_pure
# min_hist_len = 1000 # for kuairand_1k

while True:
    pick_uid = users[np.random.randint(len(users))]
    # pick_uid = 5401
    user_history = df_rand[df_rand['user_id'] == pick_uid] # & (df_rand['is_click'] == 1)
    user_timestamps = np.array(user_history['time_ms'].values) // 1000
    if len(user_timestamps) > min_hist_len:
        break
T = np.array([dt.fromtimestamp(t) for t in user_timestamps])
min_hour, max_hour = np.min(user_timestamps) / 3600, np.max(user_timestamps) / 3600
L = int(max_hour - min_hour)
plt.figure(figsize = (16,4))
plt.hist(T, bins = L + 1)
plt.yscale('log')
plt.show()
print('user id:', pick_uid, ', history length:', L)

In [ ]:
user_history['date'] = T
daily_records = user_history.resample('H', on='date').agg({'is_click':'sum', 'is_like':'sum', 
                                                 'is_follow':'sum', 'is_comment':'sum', 
                                                 'is_forward':'sum', 'is_hate':'sum', 'long_view':'sum'})

In [ ]:
# print(daily_records)
feedback_type = 'is_click'
plt.figure(figsize = (16,4))
X = np.arange(len(daily_records))
plt.bar(daily_records.index,daily_records[feedback_type],width = 0.04)
plt.show()

## Retention

In [ ]:
from datetime import datetime as dt
from tqdm import tqdm
def get_return_day_distribution(df, limit = -1):
    user_return_interval_freq = {}
    user_day_list = []
    current_uid = -1
    L = len(df) if limit < 0 else limit
    for i in tqdm(range(L)):
        row = df.iloc[i]
        t = dt.strptime(str(row['date']), "%Y%m%d")
        if row['user_id'] != current_uid:
            current_uid = row['user_id']
            user_day_list = [t]
        else:
            diff = (t - user_day_list[-1]).days
            user_day_list.append(t)
            if diff > 0:
                user_return_interval_freq[diff] = user_return_interval_freq[diff] + 1 if diff in user_return_interval_freq else 1
    return user_return_interval_freq

In [ ]:
interval_dict = get_return_day_distribution(df_rand)
# interval_dict = get_return_day_distribution(df_rand, limit = 3000000)

In [ ]:
X = list(interval_dict.keys())
print(interval_dict)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
K = np.arange(1,max(X)+1)
Y = np.array([interval_dict[k] if k in interval_dict else 0. for k in K])
Y = Y / sum(Y)
print(Y)
Z = [np.sum(Y[:i]) for i in range(1,len(Y)+1)]
plt.figure(figsize = (16,10))
plt.subplot(2,2,1)
plt.bar(K, Y)
plt.title("P(Return) @ Day")
# plt.xlabel("Day")
plt.subplot(2,2,2)
plt.bar(K, Z)
plt.plot(K, Z, color = 'black')
plt.scatter(K, Z, color = 'black')
plt.title("Return in days")
plt.xlabel("Day")
plt.subplot(2,2,3)
plt.bar(K, np.array([interval_dict[k] if k in interval_dict else 0. for k in K]))
plt.title("Return (log count) @ Day")
plt.xlabel("Day")
plt.yscale('log')
# plt.subplot(2,2,4)
# plt.bar(K, Y)
# plt.title("log P(Return) @ Day")
# plt.xlabel("Day")
# plt.yscale('log')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
observe_days = 10
P_list = [0.9, 0.7, 0.5, 0.3, 0.1]

X = np.arange(observe_days)
plt.figure(figsize = (16,4))
plt.subplot(1,2,1)
for P in P_list:
    Y = [((1-P)**i)*P for i in range(observe_days)]
    plt.plot(X,Y,label=f'P={P}')
plt.xlabel('C')
plt.ylabel('return probability')
plt.legend()
plt.subplot(1,2,2)
for P in P_list:
    Y = [((1-P)**i)*P for i in range(observe_days)]
    plt.plot(X,Y,label=f'P={P}')
plt.yscale('log')
plt.xlabel('C')
# plt.ylabel('return probability')
plt.legend()
plt.show()

In [ ]:
import torch
from torch.distributions import Beta
m = Beta(torch.ones(10000)*10,torch.tensor([1]))
H = m.sample()
print(H)
plt.hist(H.numpy(), bins = 100)
plt.ylabel("sample user count")
plt.xlabel("P")
plt.show()

## Request Time Gap

In [ ]:
from datetime import datetime as dt
from tqdm import tqdm
def get_next_request_gap_distribution(df, limit = -1):
    same_day_gap = []
    cross_day_gap = []
    user_day_list = []
    current_uid = -1
    L = len(df) if limit < 0 else limit
    for i in tqdm(range(L)):
        row = df.iloc[i]
        t = dt.fromtimestamp(row['time_ms'] // 1000)
        if row['user_id'] != current_uid:
            current_uid = row['user_id']
            user_day_list = [t]
        else:
            diff = t - user_day_list[-1]
            if diff.seconds > 0:
                if diff.days > 0:
                    cross_day_gap.append(diff.seconds / 3600.)
                else:
                    same_day_gap.append(diff.seconds / 3600.)
            user_day_list.append(t)
    return same_day_gap, cross_day_gap

In [ ]:
gap_distribution = get_next_request_gap_distribution(df_rand, limit = 1000000)

In [ ]:
plt.figure(figsize = (16,4))
plt.subplot(1,2,1)
plt.hist(gap_distribution[0], bins = 100)
plt.yscale('log')
plt.xlabel('hour')
plt.title("Same day request gap")
plt.subplot(1,2,2)
plt.hist(gap_distribution[1], bins = 100)
plt.title("Cross day request gap (mod 1 day)")
plt.xlabel('hour')
plt.show()

## Rec List Size

In [ ]:
from datetime import datetime as dt
from tqdm import tqdm
def get_list_size_distribution(df, limit = -1):
    rec_list_size = []
    n_rec_list_per_user = []
    
    user_hist = []
    current_uid = -1
    L = len(df) if limit < 0 else limit
    for i in tqdm(range(L)):
        row = df.iloc[i]
        t = dt.fromtimestamp(row['time_ms'] // 1000)
        if row['user_id'] != current_uid:
            if len(user_hist) > 0:
                n_rec_list_per_user.append(len(user_hist))
                for rec_list in user_hist:
                    rec_list_size.append(len(rec_list))
            current_uid = row['user_id']
            user_hist = [[t]]
        else:
            if user_hist[-1][-1] == t:
                user_hist[-1].append(t)
            else:
                user_hist.append([t])
        
            
    return rec_list_size, n_rec_list_per_user

In [ ]:
RLS, NLPU = get_list_size_distribution(df_rand, limit = 1000000)

In [ ]:
print(max(RLS), max(NLPU))
plt.figure(figsize = (16,4))
plt.subplot(1,2,1)
plt.hist(RLS, bins = 50)
plt.xlabel("rec list size")
plt.ylabel("sample list count")
plt.title("rec list size distribution")
plt.yscale('log')
plt.subplot(1,2,2)
plt.hist(NLPU, bins = 100)
plt.xlabel("#request")
plt.ylabel("sample user count")
plt.title("#request per user")
plt.yscale('log')
plt.show()

In [ ]:
# Build user history
from tqdm import tqdm
import numpy as np
users = list(df_rand['user_id'].unique())
position = np.zeros(len(df_rand), dtype = int)
session = np.zeros(len(df_rand), dtype = int)
session_length_list = []
history_length_list = []
n_day_list = []
user_session_length = {}
for i,uid in tqdm(enumerate(users)):
    user_data = df_rand[df_rand['user_id'] == uid]
    dates = user_data['date'].unique()
    user_session_length[uid] = []
    history_length_list.append(len(user_data))
    n_day_list.append(len(dates))
    for sess_id,D in enumerate(dates):
        day_session = user_data[user_data['date'] == D]
        session[day_session.index] = sess_id
        position[day_session.index] = np.arange(len(day_session))
        session_length_list.append(len(day_session))
        user_session_length[uid].append(len(day_session))
#         print(day_session[['user_id', 'date']])
#         print(session[day_session.index])
#         print(position[day_session.index])
#     break
    

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
plt.rcParams.update({'font.size': 16})
plt.figure(figsize = (16,8))

plt.subplot(2,2,1)
plt.hist(session_length_list, bins=100)
plt.xlabel('day session length')
plt.ylabel('frequency')
plt.yscale('log')

plt.subplot(2,2,2)
plt.hist(history_length_list, bins = 50)
plt.xlabel("user history length")
plt.ylabel("#user")
plt.yscale('log')

plt.subplot(2,2,3)
plt.hist(n_day_list, bins = 30)
plt.xlabel("number of day sessions")
plt.ylabel("#user")
plt.yscale('log')
plt.show()

In [ ]:
# Build user history
from tqdm import tqdm
import numpy as np
items = list(df_rand['video_id'].unique())
item_freq = {}
for i,iid in tqdm(enumerate(items)):
    item_data = df_rand[df_rand['video_id'] == iid]
    item_freq[iid] = len(item_data)
    

In [ ]:
import matplotlib.pyplot as plt
L = list(item_freq.values())
print(min(L), max(L))
plt.figure(figsize = (8,4))
plt.hist(L, bins = 100)
plt.yscale('log')
# plt.xscale('log')
plt.ylabel('item frequency')
plt.ylabel('item count')
plt.show()

## Immediate Responses

In [ ]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from plot_utils import plot_multiple_bars
stat_list = []
F_list = ['is_click', 'long_view', 'is_like', 'is_comment', 'is_forward', 'is_follow', 'is_hate'] # , 'is_rand'
for F in F_list:
    counts = df_rand[F].value_counts()
    print(F, counts.index, '\n', counts.values, float(counts[1]) / counts[0])
    stat_list.append((counts.index, [str(x) for x in counts.index], counts.values))
plot_multiple_bars(stat_list, F_list, ncol = 3, log_value = True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
F_list = ['is_click', 'is_like', 'is_follow', 'is_comment', 'is_forward', 'is_hate', 'long_view']
stat_dict = {k: 0 for k in F_list}
for F in F_list:
    counts = df_rand[F].value_counts()
    print(F, counts.values[1], counts.values[1] / len(df_rand))
    stat_dict[F] = counts.values[1]
sorted_stats = sorted(stat_dict.items(), key = lambda x: x[1])
sorted_keys = [k for k,v in sorted_stats]
plot_multiple_bars([(np.arange(len(F_list)), sorted_keys, [stat_dict[k] for k in sorted_keys])], 
                   ['response'], ncol = 1, row_height = 4, log_value = True, horizontal = True, font_size = 24)

In [ ]:
from tqdm import tqdm
def get_association_matrix(df, behavior_features, target_value_1 = 1, target_value_2 = 1):
#     confusion_matrix = {from_f: {to_f: 0 for to_f in behavior_features} for from_f in behavior_features}
#     association_matrix = {from_f: {to_f: 0 for to_f in behavior_features} for from_f in behavior_features}
    confusion_matrix = [[0]*len(behavior_features) for _ in behavior_features]
    association_matrix = [[0]*len(behavior_features) for _ in behavior_features]
    for i,from_f in tqdm(enumerate(behavior_features)):
        all_count = len(df[(df[from_f] == target_value_1)])
        for j,to_f in enumerate(behavior_features):
            confusion_matrix[i][j] = len(df[(df[from_f] == target_value_1) & (df[to_f] == target_value_2)])
            association_matrix[i][j] = float(confusion_matrix[i][j]) / all_count
    return confusion_matrix, association_matrix

In [ ]:
print(sorted_keys)
confusion, association = get_association_matrix(df_rand, sorted_keys, target_value_1 = 1)
print(confusion)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_heatmap(x_labels, y_labels, matrix, title = "confusion matrix", include_diagonal = True, 
                 cell_width = 2, log_scale = True, cell_str_format = "{:d}"):
    assert len(matrix) == len(y_labels) and len(matrix[0]) == len(x_labels)
    fig, ax = plt.subplots(figsize = (len(x_labels) * cell_width,len(y_labels) * cell_width))
    plot_matrix = np.log(np.array(matrix)) if log_scale else np.array(matrix)
    
    if not include_diagonal:
        for i in range(len(x_labels)):
            plot_matrix[i,i] = 0
        im = ax.imshow(plot_matrix)
    else:
        im = ax.imshow(plot_matrix)

    ax.set_xticks(np.arange(len(x_labels)))
    ax.set_yticks(np.arange(len(y_labels)))
    ax.set_xticklabels(x_labels)
    ax.set_yticklabels(y_labels)
    
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right",
             rotation_mode="anchor")
    
    for i in range(len(y_labels)):
        for j in range(len(x_labels)):
            text = ax.text(j, i, cell_str_format.format(matrix[i][j]), fontsize = 'small',
                           ha="center", va="center", color="w")
            
    ax.set_title(title)
    fig.tight_layout()
    plt.show()

In [ ]:
# plot_heatmap(sorted_keys, sorted_keys, confusion, title = 'confusion matrix', 
#              cell_width = 2, include_diagonal = True, log_scale = True)
plot_heatmap(sorted_keys, sorted_keys, association, title = 'association matrix', 
             cell_width = 2, include_diagonal = True, log_scale = True, cell_str_format = "{:.4f}")

In [ ]:
import torch
N = len(F_list)
relevance = torch.tensor(confusion).view(N,N).to(torch.float)
# print(relevance)
diag = torch.tensor([confusion[i][i] for i in range(N)]).to(torch.float)
# print(diag)
conditional_P = relevance / diag.view(N,1)
relevance = conditional_P * len(df_rand) / diag.view(1,N)

# print(relevance)
plot_heatmap(sorted_keys, sorted_keys, relevance, title = 'relevance matrix', 
             cell_width = 2, include_diagonal = False, log_scale = True, cell_str_format = "{:.2f}")

In [ ]:
import numpy as np
from plot_utils import plot_multiple_hists
 
F_list = ['play_time_ms', 'duration_ms']
hist_stat_list = []
for F in F_list:
    stat = np.array(df_rand[F].values)
    stat.sort()
    hist_stat_list.append(np.log(stat+1))
    print(min(stat), max(stat) / 3600000)
plot_multiple_hists(hist_stat_list, F_list, n_bin = 50)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from plot_utils import plot_multiple_bars
stat_list = []
F_list = ['tab']
for F in F_list:
    counts = df_rand[F].value_counts()
    print(F, counts.index, '\n', counts.values)
    stat_list.append((np.arange(len(counts.index)), [str(x) for x in counts.index], counts.values))
plot_multiple_bars(stat_list, F_list, ncol = 2, row_height = 6, log_value = True, horizontal = True)

## Video Features

In [ ]:
import pandas as pd
# video_features_basic = pd.read_csv(data_path + "video_features_basic_pure.csv", sep = '\t')
# video_features_basic = pd.read_csv(data_path + "video_features_basic_1k.csv", sep = ',')
# video_features_basic = pd.read_csv(data_path + "video_features_basic_27k.csv", sep = '\t')

video_features_basic = pd.read_csv("dataset/kuairand/kuairand-Pure/data/video_features_basic_pure.csv")
# video_features_basic = pd.read_csv(data_path + "video_features_basic_1k_fillna.csv", sep = ',')

In [ ]:
# video_features_basic['tag'] = video_features_basic['tag'].fillna(0)
# video_features_basic.to_csv(data_path + "video_features_basic_1k_fillna.csv", index = False)
# video_features_basic = pd.read_csv(data_path + "video_features_basic_1k_fillna.csv", sep = ',')

In [ ]:
video_features_basic

In [ ]:
D = video_features_basic.set_index('video_id').to_dict('index')
print(D.keys())

In [ ]:
print(D[7578])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from plot_utils import plot_multiple_bars
stat_list = []
F_list = ['visible_status', 'video_type', 'music_type'] #'upload_type'
for F in F_list:
    counts = video_features_basic[F].value_counts(dropna = False)
    print(F, counts.index, '\n', counts.values)
    stat_list.append((np.arange(len(counts.index)), [str(x) for x in counts.index], counts.values))
plot_multiple_bars(stat_list, F_list, ncol = 3, row_height = 4, log_value = True, horizontal = True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
stat_list = []
F_list = ['upload_type']
for F in F_list:
    counts = video_features_basic[F].value_counts()
    print(F, counts.index, '\n', counts.values)
    stat_list.append((np.arange(len(counts.index)), [str(x) for x in counts.index], counts.values))
plot_multiple_bars(stat_list, F_list, ncol = 1, row_height = 8, log_value = True, horizontal = True)

In [ ]:
import numpy as np
from plot_utils import plot_multiple_hists
F_list = ['video_duration', 'server_width', 'server_height']
hist_stat_list = []
for F in F_list:
    stat = np.array(video_features_basic[F].values)
    stat.sort()
    hist_stat_list.append(stat)

plot_multiple_hists(hist_stat_list, F_list, log_value = True, ncol = 3, n_bin = 50)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

counts = video_features_basic['music_id'].value_counts()
print(len(counts.index))
X = np.arange(len(counts.index))
Y = counts.values
plt.rcParams.update({'font.size': 16})
plt.loglog(X,Y)
# plt.scatter(X, np.log(Y))
plt.xlabel("Users")
plt.ylabel("Frequency")
plt.title('music_id distribution')
plt.show()

In [ ]:
from tqdm import tqdm
def get_ID_frequency(df, feature_name, separator = ','):
    ID_freq = {}
    for row in tqdm(df['tag']):
        IDs = str(row).split(separator)
        for ID in IDs:
            if ID not in ID_freq:
                ID_freq[ID] = 1
            else:
                ID_freq[ID] += 1
    return ID_freq
tag_stats = get_ID_frequency(video_features_basic, 'tag')
# print([tag_stats[k] for k in sorted_keys])

In [ ]:
sorted_stats = sorted(tag_stats.items(), key=lambda x: x[1], reverse=True)
sorted_keys = [k for k,v in sorted_stats]
stat_list = [(np.arange(len(sorted_keys)), sorted_keys, [tag_stats[k] for k in sorted_keys])]
F_list = ['tag']
plot_multiple_bars(stat_list, F_list, ncol = 1, row_height = 16, log_value = True, horizontal = True)

In [ ]:
# del tag_stats['nan']
sorted_stats = sorted(tag_stats.items(), key=lambda x: int(x[0]), reverse=True)
sorted_keys = [k for k,v in sorted_stats]
stat_list = [(np.arange(len(sorted_keys)), sorted_keys, [tag_stats[k] for k in sorted_keys])]
F_list = ['tag']
plot_multiple_bars(stat_list, F_list, ncol = 1, row_height = 16, log_value = True, horizontal = True)

## Video Statistics

In [ ]:
video_features_statistics = pd.read_csv("dataset/kuairand/kuairand-Pure/data/video_features_basic_pure.csv")

In [ ]:
video_features_statistics

In [ ]:
observe_features = list(video_features_statistics.keys())[1:]

In [ ]:
import numpy as np
 
# F_list = ['show_cnt', 'show_user_num', 'play_cnt', 'play_user_num', 'play_duration']
F_list = observe_features
hist_stat_list = []
for F in F_list:
    stat = np.array(video_features_statistics[F].values)
    stat.sort()
    hist_stat_list.append(np.log(stat+1e-6))
#     hist_stat_list.append(stat)

plot_multiple_hists(hist_stat_list, F_list, ncol = 3, n_bin = 50)

## User Features

In [ ]:
user_features = pd.read_csv(data_path + "user_features_pure.csv")
# user_features = pd.read_csv(data_path + "user_features_1k.csv")

In [ ]:
user_features

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from plot_utils import plot_multiple_bars
stat_list = []
F_list = ['user_active_degree', 'is_live_streamer', 'is_video_author', 
          'follow_user_num_range', 'fans_user_num_range', 'friend_user_num_range', 'register_days_range']
#, 'is_lowactive_period'
for F in F_list:
    counts = user_features[F].value_counts(dropna = False)
    print(F, counts.index, '\n', counts.values)
    stat_list.append((np.arange(len(counts.index)), [str(x) for x in counts.index], counts.values))
plot_multiple_bars(stat_list, F_list, ncol = 2, row_height = 6, log_value = False, horizontal = True)

In [ ]:
import torch
A = torch.randint(0,2,(5,)).to(torch.bool)
print(A.to(torch.long))
print((~A).to(torch.long))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
stat_list = []
F_list = [f'onehot_feat{fid}' for fid in range(18) if fid not in [2,3,4,5,7,8]]
for F in F_list:
    counts = user_features[F].value_counts(dropna = False)
    print(F, counts.index, '\n', counts.values)
    stat_list.append((np.arange(len(counts.index)), [str(x) for x in counts.index], counts.values))
plot_multiple_bars(stat_list, F_list, ncol = 2, row_height = 7, log_value = True, horizontal = True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
stat_list = []
F_list = [f'onehot_feat{fid}' for fid in [2,3,4,5,7,8]]
for F in F_list:
    counts = user_features[F].value_counts()
    print(F, counts.index, '\n', counts.values)
    stat_list.append((np.arange(len(counts.index)), [str(x) for x in counts.index], counts.values))
plot_multiple_bars(stat_list, F_list, ncol = 2, row_height = 6, log_value = True, horizontal = True)

In [ ]:
import numpy as np
from plot_utils import plot_multiple_hists
F_list = ['follow_user_num', 'fans_user_num', 'friend_user_num', 'register_days']
# F_list = observe_features
hist_stat_list = []
for F in F_list:
    stat = np.array(user_features[F].values)
    stat.sort()
    hist_stat_list.append(np.log(stat+1e-6))
#     hist_stat_list.append(stat)

plot_multiple_hists(hist_stat_list, F_list, ncol = 3, n_bin = 50)

# Preprocessing

In [ ]:
# output_path = '../dataset/kuairand/Kuairand-1K/'
output_path = '../dataset/kuairand/Kuairand-Pure/'

In [ ]:
import pandas as pd

# set your own data source path!

data_path = '../../dataset/kuairand/Kuairand-Pure/data/'
df_rand = pd.read_csv("dataset/kuairand/kuairand-Pure/data/log_standard_4_08_to_4_21_pure.csv")
df_rand_2 = pd.read_csv("dataset/kuairand/kuairand-Pure/data/log_standard_4_22_to_5_08_pure.csv")
df_rand_3 = pd.read_csv("dataset/kuairand/kuairand-Pure/data/log_random_4_22_to_5_08_pure.csv")
# data_path = '../../dataset/kuairand/KuaiRand-1K/data/'
# df_rand = pd.read_csv(data_path + 'log_standard_4_08_to_4_21_1k.csv')
# df_rand_2 = pd.read_csv(data_path + 'log_standard_4_22_to_5_08_1k.csv')

df_rand = pd.concat([df_rand, df_rand_2], axis=0, ignore_index=True)
df_rand = pd.concat([df_rand, df_rand_2, df_rand_3], axis=0, ignore_index=True)
df_rand

In [ ]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
from utils import run_multicore
filterred_df_rand = run_multicore(df_rand, user_key = "user_id", item_key = "video_id", 
                                  n_core = 20, auto_core = False, filter_rate = 0.2)

In [ ]:
users = list(df_rand['user_id'].unique())
print("#users:", len(users), ", #items:", len(df_rand['video_id'].unique()), ", #record:", len(df_rand))
df_rand = filterred_df_rand
print("#users:", len(df_rand['user_id'].unique()), ", #items:", len(df_rand['video_id'].unique()), ", #record:", len(df_rand))


In [ ]:
# Build user history
from tqdm import tqdm
import numpy as np

users = list(df_rand['user_id'].unique())
position = np.zeros(len(df_rand), dtype = int)
session = np.zeros(len(df_rand), dtype = int)
for i,uid in tqdm(enumerate(users)):
    user_data = df_rand[df_rand['user_id'] == uid]
    dates = user_data['date'].unique()
    for sess_id,D in enumerate(dates):
        day_session = user_data[user_data['date'] == D]
        session[day_session.index] = sess_id
        position[day_session.index] = np.arange(len(day_session))
    

In [ ]:
df_rand['position'] = position
df_rand['session'] = session
df_rand

In [ ]:
df_rand = df_rand.sort_values(by = ['user_id', 'time_ms'])
df_rand

In [ ]:
from datetime import datetime as dt
date = [int(dt.fromtimestamp(tms//1000).strftime("%Y%m%d")) for tms in df_rand['time_ms'].values]
df_rand['date'] = date

In [ ]:
output_path = '../dataset/Kuairand_Pure/'
# df_rand.to_csv('dataset/kuairand/kuairand-Pure/data/log_session_4_08_to_5_08_Pure.csv', index = False)
df_rand.to_csv('dataset/kuairand/kuairand-Pure/data/log_full_session_4_08_to_5_08_Pure.csv', index = False)

# df_rand.to_csv(output_path + 'log_session_4_08_to_5_08_1K.csv', index = False)

#### Preprocess video features

In [ ]:
import pandas as pd
video_features_basic = pd.read_csv("dataset/kuairand/kuairand-Pure/data/video_features_basic_pure.csv")
# video_features_basic = pd.read_csv(data_path + "video_features_basic_1k.csv", sep = ',')

video_features_basic['tag'] = video_features_basic['tag'].fillna(0)
video_features_basic['music_type'] = video_features_basic['music_type'].fillna(0)

video_features_basic.to_csv("dataset/kuairand/kuairand-Pure/data/video_features_basic_Pure_fillna.csv", index = False)
# video_features_basic.to_csv(output_path + "video_features_basic_1K_fillna.csv", index = False)


In [ ]:
video_features_basic

#### Preprocess user features

In [ ]:
import pandas as pd
user_features = pd.read_csv("dataset/kuairand/kuairand-Pure/data/user_features_pure.csv")
# user_features = pd.read_csv(data_path + "user_features_1k.csv", sep = ',')
for k in [f'onehot_feat{fid}' for fid in [0,1,6,9,10,11,12,13,14,15,16,17]]:
    user_features[k] = user_features[k].fillna(-1)
user_features.to_csv("dataset/kuairand/kuairand-Pure/data/user_features_Pure_fillna.csv", index = False)
# user_features.to_csv(output_path + "user_features_1K_fillna.csv", index = False)

In [ ]:
user_features